# M1.6_柱球坐标与特殊函数声辐射 — 交互式实验工作台（由 HTML 演示网页生成）

> **所属模块**：`01_M1_物理与解析基准`  
> **实验定位**：工科本科生计算物理与 SciML 交互式上机实验  
> **运行环境**：本地 Jupyter / VS Code / Colab / 个人笔记本 CPU  

---

## 本讲演示页序导览（Notebook 章节与之严格一一对应）

1. M1.6 柱/球坐标、特殊函数与声辐射
2. ① 工程场景：电机外壳散热、球罐分层与扬声器远场
3. ② 失效先行：曲面不是“把直角网格旋转一下”
4. ③ 物理直觉：换坐标系不只是“换个记号”，是三件事一起变
5. ④ 柱坐标 Laplacian 与轴对称守恒
6. ⑤ 偏心圆柱/单侧加热：Bessel 模态从哪里来
7. ⑥ 球对称热方程：用 $v=ru$ 消掉几何项
8. ⑦ 球面角向模态与 Legendre 多项式
9. ⑧ 声辐射：从近场到远场的条件
10. ⑨ 最小代码：偏心圆柱的角向展开
11. ⑩ 验证三件套与 POE 预测表
12. ⑪ 现场诊断挑战（代码面试）：看证据，找病因
13. ⑫ 本节验收清单
14. 总结与 M2 衔接

> 对齐说明：本 Notebook 由 `slides/*_演示网页.html` 自动生成；
> 代码一律从 `code/solution.py` 真实导入，函数签名由磁盘源码决定，不会漂移。


## 逐页理论与实验（与演示页序一致）

### 第 2 页 · ① 工程场景：电机外壳散热、球罐分层与扬声器远场

- 电机外壳的温升、球形储罐的径向温度分层、扬声器向外的声辐射，看起来是三个行业问题，本质却是同一件事： 几何从直角变成曲面之后，微分算子、积分测度和"远处"的边界条件都要重新写 。
- 圆柱壳 ：沿周向是否均匀取决于边界换热条件，而不是"网格画得像不像圆"；
- 球罐 ：从球心到外壳是一维径向问题，但体积元带  $r^2$  权重，通量必须按球面积分；
- 扬声器 ：真正关心的是离开本体后的出射波，计算域必须人为截断，截断边界不能简单令  $u=0$ 。
- 30 秒思考 ：把无限域的声辐射问题截成有限网格、并在外边界直接令  $u=0$ ，被丢掉的是什么物理？在圆柱轴心  $r=0$  处，直角坐标的差分模板会遇到什么麻烦？


### 第 3 页 · ② 失效先行：曲面不是“把直角网格旋转一下”

- 在圆柱或球体上直接套用笛卡尔差分，常见问题包括：边界呈阶梯状、轴心处  $1/r$  项爆炸、远场波被错误反射。
- 几何失效卡 · F3/F4 ：坐标变换改变了算子、积分测度和“远处”的边界条件——三者必须分别建基准。
- - **F3（模型/约束层）**：轴心  $r=0$  的  $1/r$  奇异与  $v(0,t)=0$  正则性条件漏列、积分测度丢掉  $2\pi r$  或  $4\pi r^2$ 、外边界写  $u=0$  造成刚性全反射。判据：**换一个更细的网格、改一个容差都不会消失**——因为错的是模型与边界条件本身。
- - **F4（离散/实现层）**：角向截断  $M$  取小、网格未加密、PML 厚度不足。判据：**加密/加密  $M$  会单调改善**。
- 两者都会表现为“近场云图好看”，但**修复手段完全不同**，混为一谈就无法归因。单元内所有  $r=0$  正则性与面积权重问题一律记 **F3**。


### 第 4 页 · ③ 物理直觉：换坐标系不只是“换个记号”，是三件事一起变

- 先不碰公式，先想清楚“变的是什么” ——
- 1. 算子变了：直角网格上的模板不能直接搬 。在平面上， $2\pi$  弧度对应一段固定长度；但绕着轴心走一圈，路程是  $2\pi r$ —— 越靠近轴心，一小格角度对应的弧长越短 。所以柱坐标里的差分必须带  $1/r$  这个权重因子。忘记它，就像用同一把尺子去量长短不一的段。
- 2. 测度变了：同样的“体积”在轴心附近少得多 。一个宽  $\Delta r$ 、高  $\Delta z$  的薄片，柱坐标里的体积是  $2\pi r\,\Delta r\,\Delta z$ ，球坐标里是  $4\pi r^2\Delta r$ 。所以 积分（也就是热量账本）必须按面积因子加权 。漏掉它，账就平了——曲线照样平滑，但总热量凭空少一大截。
- 3. “远处”的含义变了 。平面问题可以随便取大网格代表无穷远；曲面声辐射不行——你必须人为划一个边界，而 这个边界的条件决定波是走掉还是弹回来 。令  $u=0$  不是“近似无穷远”，而是 造了一面墙 。
- 所以 ：出特殊函数（ $J_m$  /  $H_m^{(1)}$  / Legendre）不是查表，而是 “区域 + 方程符号 + 轴心/无穷远条件”三者的联合筛选 ——先定区域（内还是外），再定方程（Helmholtz 还是 Laplace），最后用正则性（ $r=0$ ）或辐射条件（ $r	o\infty$ ）二选一。
- 轴心类比： $v=ru$  为什么要这么换 。球对称方程里那个讨厌的  $2u_r/r$  项，其实在做代换  $v=ru$  后 自动消失 （这是“剥掉一个  $r$ ”的标准技巧）。代价是： $u$  在轴心必须 有界 ，于是  $v=ru$  在  $r=0$  处必须 为零 。这不是随便指定的边界值，而是 由“温度不能无穷大”这条物理要求推出来的约束 。忘了它，就等于允许  $u\sim1/r$  的发散解混进答案里。


### 第 5 页 · ④ 柱坐标 Laplacian 与轴对称守恒

- 柱坐标  $(r,\theta,z)$  下
- $\nabla^2u=\frac1r\frac{\partial}{\partial r}\left(r\frac{\partial u}{\partial r}\right)+\frac1{r^2}u_{\theta\theta}+u_{zz}.$
- 轴对称且无  $z$  变化时，导热方程变为
- $\frac1r\frac{d}{dr}\left(r\frac{dT}{dr}\right)=0.$
- 积分时体积/面积权重含  $2\pi r$ ；漏掉这个  $r$ ，即使曲线看起来平滑，整体热量也不守恒。


### 第 6 页 · ⑤ 偏心圆柱/单侧加热：Bessel 模态从哪里来

- 柱坐标中对角向作 Fourier 展开
- $u(r,\theta)=\sum_{m=-\infty}^{\infty}R_m(r)e^{im\theta}.$
- Laplace/Helmholtz 方程的径向方程导出 Bessel 或修正 Bessel 函数：
- $r^2R_m''+rR_m'+(k^2r^2-m^2)R_m=0.$
- 有限圆域内选择在  $r=0$  有界的  $J_m(kr)$ ；外部辐射问题则需要满足出射条件的 Hankel 函数  $H_m^{(1)}(kr)$ 。
- “选哪个特殊函数”不是记忆题：先写区域（内/外）、方程符号和轴心/无穷远条件，再由正则性和辐射条件筛选。


### 第 7 页 · ⑥ 球对称热方程：用 $v=ru$ 消掉几何项

- 球对称扩散满足
- $u_t=\alpha\frac1{r^2}\frac{\partial}{\partial r}\left(r^2u_r\right).$
- 令  $v(r,t)=r\cdot u(r,t)$ ，则在  $r>0$  上
- $v_t=\alpha v_{rr}.$
- 但轴心仍须满足物理正则性：有限温度要求  $v(0,t)=0$ ，不能把一维区间端点随意当作普通 Dirichlet 边界。漏掉这条几何约束会使定解条件失配，属于  F3 边界条件与约束失配 。


### 第 8 页 · ⑦ 球面角向模态与 Legendre 多项式

- 轴对称球面问题的角向方程给出
- $\frac{d}{d\mu}\left[(1-\mu^2)\frac{dP_\ell}{d\mu}\right]+\ell(\ell+1)P_\ell=0,\quad \mu=\cos\theta.$
- 其解为 Legendre 多项式  $P_\ell(\cos\theta)$ ；非轴对称问题进一步使用球谐函数  $Y_\ell^m$ 。角向正交性使球面热场、静电场和声场都能复用同一模态库。


### 第 9 页 · ⑧ 声辐射：从近场到远场的条件

- 频域声压满足 Helmholtz 方程
- $\nabla^2p+k^2p=0.$
- 圆柱/球体振动产生的外部解必须选择出射 Hankel 或球 Hankel 函数，并满足 Sommerfeld 辐射条件
- $\lim_{r\to\infty}r^{(d-1)/2}(\partial_rp-ikp)=0.$
- 有限计算域中可用吸收层或 PML 模拟无穷远；出射条件写错属于  F3 边界条件与约束失配 。若外边界反射或 PML 厚度不足，近场图仍可能“好看”，但远场方向性和能量通量会错误——这正是  F4 离散截断与多尺度 “无报错但结果不合理”的典型形态。


### 第 10 页 · ⑨ 最小代码：偏心圆柱的角向展开

- Lab · 最小可运行代码
- 练习阶梯 ① 跑通并读表
- 💡  操作建议 ：先按原代码运行（ M=12 ），看边界（ r=1 ）误差  1.99e-08  与内部（ r=0.3 ） 2.67e-15  相差 7 个量级；再把  M  改成  2 ，边界误差跳到  2.08e-02 ——同一张云图上「好看」与「边界错 2%」可以同时成立。
- python · 最小代码：偏心圆柱的角向展开
- 可编辑 · Shift+Enter 运行  ↺ 重置  ▶ 运行
- import  numpy  as  np


### 第 11 页 · ⑩ 验证三件套与 POE 预测表

- 💡  POE 自检台 ：先独立思考（可点击填写预测），再点击右侧按钮揭晓标准解析对照。
- 👁️ 全部揭晓 / 隐藏
- 检查/改动
- 运行前预测
- 观测证据（本机实测）
- 解释与失效归类


### 第 12 页 · ⑪ 现场诊断挑战（代码面试）：看证据，找病因

- 故障一 · 特殊函数选择的证据链 ：偏心圆柱声辐射与球坐标热方程——直接套用平面波展开，或忽略  r=0  正则条件；远场指向性与通量都对不上解析基准，近场云图却「好看」。
- 故障二 · 截断外边界上的「假回音壁」 ：圆柱声辐射远场预报——计算域取半径  R=5a ，外边界直接令  u=0 ；算出的远场指向性与实测相差 30% 以上，且把  R  加大后结果仍在变。
- 课程立场 ：AI 可以帮你展开级数、换求解器、调代码，但不能替你证明坐标奇异的正则性与截断边界的辐射条件—— 先判 F3 还是 F4，再选修复手段 。


### 第 13 页 · ⑫ 本节验收清单

- 能写出柱坐标 Laplacian 并解释  $r$  权重；
- 能说明  $J_m$ 、修正 Bessel、Hankel 分支的区域选择依据；
- 能用  $v=ru$  化简球对称扩散并处理轴心正则性；
- 能解释声辐射的出射条件和 PML 的作用；
- 将圆柱/球体基准整理到  benchmarks/cylindrical_spherical/ ；
- 给出至少一个远场能量通量或边界残差证据；


### 第 14 页 · 总结与 M2 衔接

- 曲线坐标把几何、正则性和特殊函数绑在一起；声辐射还要求对无穷远边界负责。完成 M1 后，课程进入 M2 数值离散：将这些解析基准用于差分、有限体积和有限元的误差、稳定性与守恒验证。
- 下一站预告 · M2.1 五点格式、稀疏矩阵与 MMS ：从本节的柱/球坐标解析基准回到二维偏微分方程的有限差分离散——用 Kronecker 积构造五点格式稀疏矩阵，以制造解（MMS）实测收敛阶 2，并诊断 Neumann 一阶近似使收敛阶降为 1（F4）等失效。本节的  $r$  权重与截断残差证据，将成为检验二维离散守恒性的对照基准。


## 演示页代码实验台
下面单元格直接抽取自演示网页的可运行代码（`cell-src`），按顺序执行即可复现网页上的交互结果。

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

def modal_field(r, theta, eps=0.25, M=12):
    """角向 Fourier 展开 u(r,θ) = Σ_{m≤M} ε^m r^m cos(mθ)（r^m 是径向衰减因子）。"""
    out = np.zeros_like(r, dtype=float)
    for m in range(M + 1):
        out += (eps ** m) * (r ** m) * np.cos(m * theta)
    return out

# --- 可检查数字①：截断误差按 r^m 集中在边界，向内迅速衰减 ---
MREF = 80                      # 足够高阶，当作“精确解”
th = np.linspace(0.0, 2*np.pi, 720, endpoint=False)
print(f"{'M':>4}{'r=1 误差':>14}{'r=0.8':>14}{'r=0.5':>14}{'r=0.3':>14}")
for M in (2, 4, 6, 8, 12):
    row = []
    for rv in (1.0, 0.8, 0.5, 0.3):
        r = np.full_like(th, rv)
        err = np.max(np.abs(modal_field(r, th, M=M) - modal_field(r, th, M=MREF)))
        row.append(err)
    print(f"{M:>4}" + "".join(f"{v:>14.3e}" for v in row))
  

## 加载本单元真实源码并核对函数签名
下面直接导入 `code/solution.py`（**不是副本**）；若你在磁盘上改了代码，这里的签名会立刻跟着变。

In [ ]:
# --- 先定位本单元目录：从 notebooks/ 或项目根目录启动 Jupyter 都能找到 code/solution.py
from pathlib import Path
import os
import sys

search_roots = (Path.cwd(), *Path.cwd().parents)
code_candidates = [parent / "code" for parent in search_roots]
for root in search_roots:
    code_candidates.extend(root.glob("*/*/code"))
code_candidates = list(dict.fromkeys(p.resolve() for p in code_candidates))
found = [p.resolve() for p in code_candidates if (p / "solution.py").is_file()]
# 从项目根目录启动时会有多个单元命中，优先锁定本单元（M1.6），避免误载他单元的 solution.py
preferred = [p for p in found if any("M1.6" in part for part in p.parts)]
CODE_DIR = (preferred or found or [None])[0]
if CODE_DIR is None:
    raise FileNotFoundError("找不到 code/solution.py，请从本单元目录或项目根目录启动 Jupyter")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
ASSET_DIR = CODE_DIR.parent / "assets"
print("CODE_DIR  =", CODE_DIR)
print("ASSET_DIR =", ASSET_DIR)


In [ ]:
import sys, inspect, importlib.util, os
SOL_PATH = os.path.join(CODE_DIR, "solution.py")
MODULE_NAME = "course_unit_M1_6_solution"
spec = importlib.util.spec_from_file_location(MODULE_NAME, SOL_PATH)
S = importlib.util.module_from_spec(spec)
# Make paths relative to the owning unit so main() is independent of Jupyter's cwd.
_OLD_CWD = os.getcwd()
try:
    os.chdir(CODE_DIR.parent)
    spec.loader.exec_module(S)
finally:
    os.chdir(_OLD_CWD)
sys.modules[MODULE_NAME] = S
print("已从磁盘加载:", SOL_PATH)
print()
def _show():
    import ast
    tree = ast.parse(open(SOL_PATH, encoding="utf-8").read())
    for n in tree.body:
        if isinstance(n, ast.FunctionDef) and not n.name.startswith("_"):
            print("def %s" % n.name, inspect.signature(getattr(S, n.name)))
        elif isinstance(n, ast.ClassDef):
            print("class", n.name)
            for m in n.body:
                if isinstance(m, ast.FunctionDef) and (not m.name.startswith("_") or m.name == "__init__"):
                    print("    ", m.name, [a.arg for a in m.args.args])
_show()

## 运行主实验
执行下方单元格即调用本单元真实入口 `main()`，输出与演示网页一致。

In [ ]:
# 执行本单元真实入口函数（网页上的数字由此产生）
try:
    S.main()
except Exception as e:
    print("运行未成功，请把完整报错发给老师：")
    import traceback; traceback.print_exc()

## 物理场成果图展示（相对路径，换机可用）

In [ ]:
from IPython.display import Image, display
import glob
pngs = sorted(glob.glob(os.path.join(ASSET_DIR, "*.png")))
if not pngs:
    print("assets 目录下暂无 png；请先运行上面的求解入口生成图表。")
for f in pngs:
    print("展示:", os.path.basename(f))
    display(Image(filename=f))